In [1]:
import pandas as pd
import numpy as np
from collections import Counter

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

In [3]:
def lineages(x, links_dataframe):
    # Parse lineage relationships between splitting events
    source_id = x.Source_ID # each row's source spot ID
    # add source id to list if the row is a splitting event
    if x.Splitting_event:
        lineage = [str(source_id)]
    else:
        lineage = []
    while True:
        target_id = links_dataframe.loc[links_dataframe['Target_ID'] == source_id,:] # find row containing corresponding target ID
        if target_id.empty:
            break
        if target_id.Splitting_event.values[0]:
            lineage.append(str(target_id.Source_ID.values[0]))
        source_id = target_id.Source_ID.values[0]
    if lineage:    
        return ".".join(reversed(lineage)) # insert . between every element in the list, in reverse order
    else:
        return None

def get_mother(x):
    lineage_list = x.split(".")
    if len(lineage_list) == 1:
        return None
    else:
        return lineage_list[-2] 

def get_grandmother(x):
    lineage_list = x.split(".")
    if len(lineage_list) < 3:
        return None
    else:
        return lineage_list[-3]     
    
def get_sister(x, dataframe):
    if x.Mother_ID is None:
        return None
    else:
        sister_df = dataframe.loc[
            (dataframe.Mother_ID == x.Mother_ID) & 
        #    (dataframe.Position == x.Position) &
            (dataframe.Source_ID != x.Source_ID), :
        ]
        if sister_df.shape[0] > 0:
            lineage_sister = sister_df.iloc[0]['Lineage']
            sister = lineage_sister.split(".")[-1]
            return sister
        else:
            pass

def get_daughters(x, dataframe):
    daughters_df = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Track_ID == x.Track_ID) &
            (dataframe.Mother_ID == x.Source_ID) & 
            (dataframe.Position == x.Position), :
    ]
    daughters_list = daughters_df.Source_ID.unique()
    return daughters_list

def get_mother_n_of_daughters(x, dataframe):

    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID) &
            (dataframe.Position == x.Position)
        ]
        
        n_daughters = mother_row.Number_of_daughters_in_mitosis.item()
        return n_daughters

def seensister(x, already_seen_list):
    if x.Sister_ID is None:
            return None
    else:
        if x.Mother_ID in already_seen_list:
            return True
        else:
            already_seen_list.append(x.Mother_ID)
            return False

def get_cousin(x, dataframe):
    if x.Grandmother_ID is None:
        return None
    else:
        cousins_df = dataframe.loc[
            (dataframe.Grandmother_ID == x.Grandmother_ID) & 
            (dataframe.Source_ID != x.Source_ID) &
            (dataframe.Mother_ID != x.Mother_ID), :
        ]
        cousins = []
        if cousins_df.shape[0] == 2:   
            cousinA = cousins_df.iloc[0, 1].item()
            cousinB = cousins_df.iloc[1, 1].item()
            return cousinA, cousinB
        elif cousins_df.shape[0] == 1:
            cousinA = cousins_df.iloc[0, 1].item()
            return cousinA
        else:
            return None

def get_random(x, dataframe):
    random_df = dataframe.loc[
            (dataframe.Position == x.Position) &
            (dataframe.Track_ID != x.Track_ID) & # Don't pair within same lineage
            (dataframe.Source_ID != x.Sister_ID) & 
            (dataframe.Source_ID != x.Mother_ID)  
           # (dataframe.Source_ID != x.Grandmother_ID) & 
           # (dataframe.Source_ID != x.Cousin_ID) & 
           # (dataframe.Source_ID != x.Aunt_ID), :
        ]
    
    random_ID = random_df["Source_ID"].sample(1)
    return random_ID.item()

def seengranny(x, granny_seen_list):
    if x.Grandmother_ID is None:
        return None
    else:
        if x.Grandmother_ID not in granny_seen_list:
            granny_seen_list.append(x.Grandmother_ID)
            return False
        else:
            return True

def get_cell_cycle(x, dataframe):
    daughter_time = x.Frame
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
           # (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            cell_cycle = (int(daughter_time) - int(mother_time)) * 7
            return cell_cycle
        else:
            return None


          
def tracking_dataframes(edges_dir, spots_dir, dataset, position, outdir):
    links = pd.read_csv(
        edges_dir,
        usecols = ['TRACK_ID', 'SPOT_SOURCE_ID', 'SPOT_TARGET_ID']
    )
    links.rename(columns = {
        "TRACK_ID": "Track_ID", 
        "SPOT_SOURCE_ID": "Source_ID", 
        "SPOT_TARGET_ID": "Target_ID"
    }, inplace = True)
    
    spots = pd.read_csv(
        spots_dir,
        usecols = ['ID', 'TRACK_ID', 'POSITION_X', 'POSITION_Y', 'FRAME']
    )
    spots.rename(columns = {
        "TRACK_ID": "Track_ID", 
        "POSITION_X": "Track_Coordinate_X", 
        "POSITION_Y": "Track_Coordinate_Y", 
        "FRAME": "Frame"
    }, inplace = True)
    
    # Generate a list of spot_ids that correspond to a splitting event
    # (Per definition, a splitting event is labelled twice as "source")
    source_ids = list(links["Source_ID"])
    source_id_counts = Counter(source_ids)
    splitting_event_ids = [id for id in source_id_counts if source_id_counts[id] > 1]
    
    # Add Boolean to Spots and Links dataframes
    # that indicate whether the spot or links belongs to a 
    # splitting event

    spots["Splitting_event"] = spots["ID"].apply(lambda x:\
                                                 False if x not in splitting_event_ids\
                                                 else True)

    links["Splitting_event"] = links["Source_ID"].apply(lambda x:\
                                                 False if x not in splitting_event_ids\
                                                 else True)

    

    links['Lineage'] = links.apply(lineages, args = (links,), axis = 1)
    print("Successfully annotated lineage information.")

    links = links[links["Splitting_event"] == True]
    spots = spots[spots["Splitting_event"] == True]
    spots.rename(columns = {"ID": "Source_ID"}, inplace = True)
    
    df = pd.merge(links, spots, how = "outer", on = ["Source_ID", "Track_ID", "Splitting_event"])
    
    df.drop(["Target_ID"], axis = 1, inplace = True)
    df.drop_duplicates(subset = ['Source_ID'], inplace = True)

    df["Position"] = position
    
    df["Dataset"] = dataset
  
    df["Generation"] = df["Lineage"].apply(lambda x: x.count(".") + 1)
    df["Mother_ID"] = df["Lineage"].apply(lambda x: get_mother(x))
    df["Grandmother_ID"] = df["Lineage"].apply(lambda x: get_grandmother(x))
    df["Sister_ID"] = df.apply(get_sister, dataframe = df, axis = 1)

    df["Daughters_ID"] = df.apply(get_daughters, dataframe = df, axis = 1)
    df["Number_of_daughters_in_mitosis"] = df['Daughters_ID'].str.len()
    df["Mother_Number_of_daughters_in_mitosis"] = df.apply(get_mother_n_of_daughters, dataframe = df, axis = 1)
    
    df["Cell_Cycle_mins"] = df.apply(get_cell_cycle, dataframe = df, axis = 1)
    df["Cell_Cycle_hours"] = df.Cell_Cycle_mins / 60

    # Randomising order of dataframe to 
    # avoid that sisters with shortest cell cycles
    # are 'False' for Seen_sister or seen_granny.
    df = df.sample(frac = 1) # shuffle rows
    
    # This will allow to sample one cell out of sister pair
    seen_sister_list = []
    print("Populating sister list")
    df["Seen_sister"] = df.apply(seensister, already_seen_list = seen_sister_list, axis = 1)    
    df["Random_ID"] = df.apply(get_random, dataframe = df, axis = 1)
    
    # This will allow to sample one cell out of cousin quartett
    seen_granny_list = []
    print("Populating granny list")
    df["Seen_granny"] = df.apply(seengranny, granny_seen_list = seen_granny_list, axis = 1) 
    
    # sort for downstream analysis after shuffling
    df = df.sort_values(['Frame', 'Track_Coordinate_X'])
    destination = outdir + "_lineages.csv"
    df.to_csv(destination)
    print("Successfully exported lineage dataframe to " + destination)
    return df

In [4]:
df_01 = tracking_dataframes(
    edges_dir = root + "20250627/Position_3_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_edges.csv", 
    spots_dir = root + "20250627/Position_3_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_spots.csv",
    dataset = "20250627",
    position = 3,
    outdir = root + "20250627/Position_3_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02"
)

df_02 = tracking_dataframes(
    edges_dir = root + "20250627/Position_4_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_edges.csv", 
    spots_dir = root + "20250627/Position_4_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_spots.csv",
    dataset = "20250627",
    position = 4,
    outdir = root + "20250627/Position_4_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02"
)

df_03 = tracking_dataframes(
    edges_dir = root + "20250704/Position_1_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_edges.csv", 
    spots_dir = root + "20250704/Position_1_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_spots.csv",
    dataset = "20250704",
    position = 1,
    outdir = root + "20250704/Position_1_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03"
)

df_04 = tracking_dataframes(
    edges_dir = root + "20250704/Position_2_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_edges.csv", 
    spots_dir = root + "20250704/Position_2_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_spots.csv",
    dataset = "20250704",
    position = 2,
    outdir = root + "20250704/Position_1_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03"
)

df_05 = tracking_dataframes(
    edges_dir = root + "20250704/Position_3_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_edges.csv", 
    spots_dir = root + "20250704/Position_3_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_spots.csv",
    dataset = "20250704",
    position = 3,
    outdir = root + "20250704/Position_3_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03"
)

df_06 = tracking_dataframes(
    edges_dir = root + "20250704/Position_4_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_edges.csv", 
    spots_dir = root + "20250704/Position_4_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_spots.csv",
    dataset = "20250704",
    position = 4,
    outdir = root + "20250704/Position_4_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03"
)

df_07 = tracking_dataframes(
    edges_dir = root + "20250704/Position_5_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_edges.csv", 
    spots_dir = root + "20250704/Position_5_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_spots.csv",
    dataset = "20250704",
    position = 5,
    outdir = root + "20250704/Position_5_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03"
)

df_08 = tracking_dataframes(
    edges_dir = root + "20250704/Position_6_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_edges.csv", 
    spots_dir = root + "20250704/Position_6_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_spots.csv",
    dataset = "20250704",
    position = 6,
    outdir = root + "20250704/Position_6_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03"
)

df_09 = tracking_dataframes(
    edges_dir = root + "20250704/Position_7_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_edges.csv", 
    spots_dir = root + "20250704/Position_7_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_spots.csv",
    dataset = "20250704",
    position = 7,
    outdir = root + "20250704/Position_7_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03"
)

Successfully annotated lineage information.
Populating sister list
Populating granny list
Successfully exported lineage dataframe to /Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/20250627/Position_3_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_lineages.csv
Successfully annotated lineage information.
Populating sister list
Populating granny list
Successfully exported lineage dataframe to /Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/20250627/Position_4_MonastrolWashout/Concatenated/20250627_IM_H2B-GFP_MitoStop_02_lineages.csv
Successfully annotated lineage information.
Populating sister list
Populating granny list
Successfully exported lineage dataframe to /Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/20250704/Position_1_MonastrolWashout/Concatenated/20250704_IM_H2B-GFP_MitoStop_03_lineages.csv
Successfully annotated lineage information.
Populating sister list
Populating granny list
Successfully exported lineage dataframe to /Volumes/ArxivE/_Tobias/LSM980/Mitot

In [5]:
dataframes = [
    df_01,
    df_02,
    df_03,
    df_04,
    df_05,
    df_06,
    df_07,
    df_08,
    df_09
]

In [6]:
df_concat = pd.concat(dataframes)
df_concat.to_csv(outputFolder + "MainDataFrame_Lineages_Monastrol.csv")

In [7]:
from statistics import mean

def get_lineage_statistics(dataframe = dataframes, outDir = outputFolder):
    
    dataframes_for_concat = []
        
    for data in dataframe:
        dataset = data.loc[0, "Dataset"]
        position = data.loc[0, "Position"]
        max_generation = data.Generation.max()
        No_of_source_ids = data.Source_ID.dropna().shape[0]
        No_of_random_ids = data.Random_ID.dropna().shape[0]
        No_of_sister_ids = data.Sister_ID.dropna().shape[0]
        No_of_mother_ids = data.Mother_ID.dropna().shape[0]
        #No_of_aunt_ids = data.Aunt_ID.dropna().shape[0]
        #No_of_cousin_ids = data.Cousin_ID.dropna().shape[0]
        No_of_grandmother_ids = data.Grandmother_ID.dropna().shape[0]
        
        list_of_tracks = data.Track_ID.unique()
        list_of_trackSubData = []
        for track in list_of_tracks:
            sub_data = data.loc[data.Track_ID == track]
            list_of_trackSubData.append(sub_data)

        No_of_tracks = data.Track_ID.nunique()
        Average_No_of_splitting_events = data.groupby("Track_ID").describe().count().mean()

        sister_per_split = []
        mother_per_split = []
        grandmother_per_split = []
        #aunt_per_split = []
        #cousins_per_split = []

        for sub_data in list_of_trackSubData: 
            number_of_sister_ids = sub_data.Sister_ID.dropna().shape[0]
            sis_per_total_splits = number_of_sister_ids / sub_data.shape[0]
            sister_per_split.append(sis_per_total_splits)

            number_of_mother_ids = sub_data.Mother_ID.dropna().shape[0]
            mom_per_total_splits = number_of_mother_ids / sub_data.shape[0]
            mother_per_split.append(mom_per_total_splits)

            number_of_gmother_ids = sub_data.Grandmother_ID.dropna().shape[0]
            gmom_per_total_splits = number_of_gmother_ids / sub_data.shape[0]
            grandmother_per_split.append(gmom_per_total_splits)

            #number_of_aunt_ids = sub_data.Aunt_ID.dropna().shape[0]
            #aunt_per_total_splits = number_of_aunt_ids / sub_data.shape[0]
            #aunt_per_split.append(aunt_per_total_splits)

            #number_of_cousin_ids = sub_data.Cousin_ID.dropna().shape[0]
            #cousin_per_total_splits = number_of_cousin_ids / sub_data.shape[0]
            #cousins_per_split.append(cousin_per_total_splits)

        statistics_dict = {"Dataset": dataset, 
                           "Position": position, 
                           "Number_of_tracks": No_of_tracks,
                           "Max_No_of_Generations": max_generation,
                           "Number_of_Source_IDs": No_of_source_ids,
                           "Number_of_Sister_IDs": No_of_sister_ids,
                           #"Number_of_Cousin_IDs": No_of_cousin_ids,
                           "Number_of_Mother_IDs": No_of_mother_ids,
                           #"Number_of_Aunt_IDs": No_of_aunt_ids,
                           "Number_of_Grandmother_IDs": No_of_grandmother_ids,
                           "Number_of_Random_IDs": No_of_random_ids,
                           "Average_No_of_SplittingEvents_per_Track": data.shape[0] / No_of_tracks, 
                           "Average_No_of_Sisters_per_Splitting_event": mean(sister_per_split), 
                           "Average_No_of_Mothers_per_Splitting_event": mean(mother_per_split), 
                           "Average_No_of_Grandmothers_per_Splitting_event": mean(grandmother_per_split), 
                           #"Average_No_of_Aunts_per_Splitting_event": mean(aunt_per_split), 
                           #"Average_No_of_Cousin_per_Splitting_event": mean(cousins_per_split)
                          }
        df = pd.DataFrame(statistics_dict, index = [0])
        dataframes_for_concat.append(df)
    
    final_df = pd.concat(dataframes_for_concat)
    final_df.to_csv(outDir + "MetaStatistics_TrackingLineages_Monastrol.csv")
    return final_df

statistics_df = get_lineage_statistics()
statistics_df

,Dataset,Position,Number_of_tracks,Max_No_of_Generations,Number_of_Source_IDs,Number_of_Sister_IDs,Number_of_Mother_IDs,Number_of_Grandmother_IDs,Number_of_Random_IDs,Average_No_of_SplittingEvents_per_Track,Average_No_of_Sisters_per_Splitting_event,Average_No_of_Mothers_per_Splitting_event,Average_No_of_Grandmothers_per_Splitting_event
0,20250627,3,37,3,97,44,59,21,97,2.621622,0.244273,0.410940,0.088674
0,20250627,4,38,3,103,52,65,25,103,2.710526,0.310276,0.419925,0.121617
0,20250704,1,8,3,19,8,11,2,19,2.375000,0.266667,0.454167,0.050000
0,20250704,2,17,3,34,10,17,4,34,2.000000,0.166667,0.328431,0.058824
0,20250704,3,28,3,74,36,46,13,74,2.642857,0.315986,0.476701,0.081888
0,20250704,4,15,3,38,20,23,7,38,2.533333,0.321587,0.404921,0.081429
0,20250704,5,10,3,32,18,22,10,32,3.200000,0.362381,0.504048,0.195476
0,20250704,6,20,4,66,38,46,20,66,3.300000,0.391429,0.522381,0.167143
0,20250704,7,10,3,31,18,21,10,31,3.100000,0.349048,0.440714,0.172143
